# 08. Experiencia externa

Proyecto de tesis (Maestria en Ciencia de Datos e IA, ESPOL): *Sistema de Generacion de Perfiles del Personal Docente y Administrativo en ESPOL para la asignacion inteligente de tareas*. Este notebook corresponde a la **Fase 1 y 2** de la metodologia: extraccion y depuracion de una de las fuentes institucionales que alimentan el catalogo de variables (historia laboral, capacitaciones, proyectos, experiencia y direcciones de tesis) usado para construir los perfiles multidimensionales del personal.

**Fuente:** `data/raw/experenciaexterna.csv`  
**Salida:** `data/processed/experiencia_externa.csv`

Historial laboral del personal fuera de ESPOL (cargo, institucion, fechas, tipo de relacion laboral). Los codigos CATEXPERIENCIA y ROLACADEMICOEXPERIENCIA se decodifican usando data/raw/diccionarioexperienciaexterna.txt.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [2]:
df = pc.leer_csv('experenciaexterna.csv', low_memory=False)
df.head()

Leido experenciaexterna.csv con encoding=utf-8-sig -> 13818 filas, 21 columnas


,IDHISTORIALABORAL,IDPERSONA,CARGO,INSTITUCION,TIPOINSTITUCION,RELACIONLABORAL,FECHADESDE,FECHAHASTA,PAIS,IDPAIS,TIEMPODEDICACION,REFEVIDENCIA,REFEVALUACIONES,CATEXPERIENCIA,CATEGORIAEXPERIENCIADESCRIPCION,ROLACADEMICO,ROLACADEMICOEXPERIENCIA,VIGENTE,TIPOPONENCIA,NOMBRECONGRESO,MODALIDAD
0,8732,660654,AYUDANTE DE INVESTIGACIÓN,ESCUELA POLITÉCNICA NACIONAL,PÚBLICA,CONTRATO CON RELACIÓN DE DEPENDENCIA,2018-07-02,2019-08-31,ECUADOR,1,MEDIO TIEMPO,"456599,459052",NaN,PC,POR CLASIFICAR,NaN,NaN,0,NaN,NaN,NaN
1,87,1614,INGENIERA DE MANTENIMIENTO,FABRICACIONES ELECTROMECÁNICAS FETL S.A.,PRIVADA,CONTRATO SIN RELACIÓN DE DEPENDENCIA,1996-01-02,2000-11-30,ECUADOR,1,MEDIO TIEMPO,NaN,NaN,PC,POR CLASIFICAR,NaN,NaN,0,NaN,NaN,NaN
2,11151,669877,ingeniero civil,celec,PÚBLICA,CONTRATO SIN RELACIÓN DE DEPENDENCIA,2018-08-03,2018-12-11,ECUADOR,1,MEDIO TIEMPO,"1012396,956161",NaN,PC,POR CLASIFICAR,NaN,NaN,0,NaN,NaN,NaN
3,295,1373,DOCENTE,UNIV. POLITECNICA SALESIANA,PRIVADA,CONTRATO CON RELACIÓN DE DEPENDENCIA,2008-06-01,2010-09-30,ECUADOR,1,MEDIO TIEMPO,"115174,114051",NaN,PC,POR CLASIFICAR,NaN,NaN,0,NaN,NaN,NaN
4,2903,1108,NaN,UNIVERSIDAD POLITÉCNICA SALESIANA,NaN,NaN,2015-12-14,NaN,ECUADOR,1,MEDIO TIEMPO,"51882,50863","51883,50864",PC,POR CLASIFICAR,NaN,NaN,0,NaN,NaN,NaN


## 2. Exploración inicial

In [3]:
pc.resumen(df, 'experiencia_externa')

--- Resumen experiencia_externa ---
Dimensiones: 13818 filas x 21 columnas
Filas duplicadas: 0
Columnas con nulos (%):
MODALIDAD                  100.0
TIPOPONENCIA               100.0
NOMBRECONGRESO             100.0
REFEVALUACIONES             98.3
ROLACADEMICOEXPERIENCIA     90.9
ROLACADEMICO                90.9
REFEVIDENCIA                26.1
RELACIONLABORAL             20.6
CARGO                        9.4
FECHAHASTA                   8.8
TIPOINSTITUCION              4.7
INSTITUCION                  0.2
FECHADESDE                   0.1
dtype: float64


In [4]:
df.dtypes

IDHISTORIALABORAL                    int64
IDPERSONA                            int64
CARGO                                  str
INSTITUCION                            str
TIPOINSTITUCION                        str
RELACIONLABORAL                        str
FECHADESDE                             str
FECHAHASTA                             str
PAIS                                   str
IDPAIS                               int64
TIEMPODEDICACION                       str
REFEVIDENCIA                           str
REFEVALUACIONES                        str
CATEXPERIENCIA                         str
CATEGORIAEXPERIENCIADESCRIPCION        str
ROLACADEMICO                           str
ROLACADEMICOEXPERIENCIA                str
VIGENTE                              int64
TIPOPONENCIA                       float64
NOMBRECONGRESO                     float64
MODALIDAD                          float64
dtype: object

## 3. Limpieza

In [5]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df)

Columnas eliminadas por tener >= 99% de nulos: ['TIPOPONENCIA', 'NOMBRECONGRESO', 'MODALIDAD']


## 5. Tipado de fechas e identificadores

In [6]:
df = pc.castear_fechas(df, ['FECHADESDE', 'FECHAHASTA'])
df = pc.castear_enteros(df, ['IDHISTORIALABORAL', 'IDPERSONA', 'IDPAIS'])

## 6. Decodificación de catálogos

Códigos definidos en `data/raw/diccionarioexperienciaexterna.txt`: CATEXPERIENCIA (PC=Por clasificar, AD=Administrativa, AC=Académica) y ROLACADEMICOEXPERIENCIA (PR=Profesor, FA=Facilitador, PA=Personal de apoyo, AY=Ayudante).

In [7]:
df = pc.decodificar_experiencia_externa(df)
df[['CATEXPERIENCIA', 'CATEXPERIENCIA_DESC', 'ROLACADEMICOEXPERIENCIA', 'ROLACADEMICOEXPERIENCIA_DESC']].drop_duplicates().head(10)

,CATEXPERIENCIA,CATEXPERIENCIA_DESC,ROLACADEMICOEXPERIENCIA,ROLACADEMICOEXPERIENCIA_DESC
0,PC,POR CLASIFICAR,<NA>,NaN
6,AD,ADMINISTRATIVA,<NA>,NaN
39,AC,ACADEMICA,PROFESOR,NaN
123,AC,ACADEMICA,<NA>,NaN
360,AC,ACADEMICA,PERSONAL DE APOYO,NaN
390,AD,ADMINISTRATIVA,PERSONAL DE APOYO,NaN
542,AC,ACADEMICA,AYUDANTE,NaN
815,AC,ACADEMICA,FACILITADOR,NaN
2693,AD,ADMINISTRATIVA,PROFESOR,NaN


## 7. Verificación final

In [8]:
pc.resumen(df, 'experiencia_externa (procesado)')
df.head()

--- Resumen experiencia_externa (procesado) ---
Dimensiones: 13818 filas x 20 columnas
Filas duplicadas: 0
Columnas con nulos (%):
ROLACADEMICOEXPERIENCIA_DESC    100.0
REFEVALUACIONES                  98.3
ROLACADEMICOEXPERIENCIA          90.9
ROLACADEMICO                     90.9
REFEVIDENCIA                     26.1
RELACIONLABORAL                  20.6
CARGO                             9.4
FECHAHASTA                        8.8
TIPOINSTITUCION                   4.7
INSTITUCION                       0.2
FECHADESDE                        0.1
dtype: float64


,IDHISTORIALABORAL,IDPERSONA,CARGO,INSTITUCION,TIPOINSTITUCION,RELACIONLABORAL,FECHADESDE,FECHAHASTA,PAIS,IDPAIS,TIEMPODEDICACION,REFEVIDENCIA,REFEVALUACIONES,CATEXPERIENCIA,CATEGORIAEXPERIENCIADESCRIPCION,ROLACADEMICO,ROLACADEMICOEXPERIENCIA,VIGENTE,CATEXPERIENCIA_DESC,ROLACADEMICOEXPERIENCIA_DESC
0,8732,660654,AYUDANTE DE INVESTIGACIÓN,ESCUELA POLITÉCNICA NACIONAL,PÚBLICA,CONTRATO CON RELACIÓN DE DEPENDENCIA,2018-07-02,2019-08-31,ECUADOR,1,MEDIO TIEMPO,"456599,459052",<NA>,PC,POR CLASIFICAR,<NA>,<NA>,0,POR CLASIFICAR,NaN
1,87,1614,INGENIERA DE MANTENIMIENTO,FABRICACIONES ELECTROMECÁNICAS FETL S.A.,PRIVADA,CONTRATO SIN RELACIÓN DE DEPENDENCIA,1996-01-02,2000-11-30,ECUADOR,1,MEDIO TIEMPO,<NA>,<NA>,PC,POR CLASIFICAR,<NA>,<NA>,0,POR CLASIFICAR,NaN
2,11151,669877,ingeniero civil,celec,PÚBLICA,CONTRATO SIN RELACIÓN DE DEPENDENCIA,2018-08-03,2018-12-11,ECUADOR,1,MEDIO TIEMPO,"1012396,956161",<NA>,PC,POR CLASIFICAR,<NA>,<NA>,0,POR CLASIFICAR,NaN
3,295,1373,DOCENTE,UNIV. POLITECNICA SALESIANA,PRIVADA,CONTRATO CON RELACIÓN DE DEPENDENCIA,2008-06-01,2010-09-30,ECUADOR,1,MEDIO TIEMPO,"115174,114051",<NA>,PC,POR CLASIFICAR,<NA>,<NA>,0,POR CLASIFICAR,NaN
4,2903,1108,<NA>,UNIVERSIDAD POLITÉCNICA SALESIANA,<NA>,<NA>,2015-12-14,NaT,ECUADOR,1,MEDIO TIEMPO,"51882,50863","51883,50864",PC,POR CLASIFICAR,<NA>,<NA>,0,POR CLASIFICAR,NaN


## 8. Guardado en data/processed

In [9]:
pc.guardar_procesado(df, 'experiencia_externa.csv')

Guardado: D:\Proyecto_Tesis\data\processed\experiencia_externa.csv (13818 filas x 20 columnas)


WindowsPath('D:/Proyecto_Tesis/data/processed/experiencia_externa.csv')